# 9.2 Publication Text Analysis - step 2: prepare text corpus and lab-level data

This notebook builds the publication-level text corpus (title + abstract combined, one row per publication-lab pair). It also creates the lab-level subject-code benchmark shares and merges these benchmark shares with BL energy use and other control vars.

In [1]:
# Set up
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data
publications = pd.read_csv(
    config.PUBLICATON_DATA / 
    "2_Processed" / 
    "publications_matched.csv"
)

labs = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Explode publications to one row per publication-lab

In [3]:
publications["pub_id"] = publications.index  # index publications before exploding

# One row per publication-lab pair
publications["labgroupid"] = publications["matched_labgroupids"].astype(str).str.split(";")
pub_long = publications.explode("labgroupid")
pub_long["labgroupid"] = pub_long["labgroupid"].str.strip()

print(f"Matched publications (before exploding): {publications.shape[0]:,}")
print(f"Publication-lab rows (after exploding): {pub_long.shape[0]:,}")
print(f"Distinct labs: {pub_long['labgroupid'].nunique()}")

Matched publications (before exploding): 8,014
Publication-lab rows (after exploding): 8,435
Distinct labs: 95


## (2) Combine title and description (abstract)

In [4]:
# Combine title and description into a single text field
def combine_text(row):
    parts = [row["title"]]
    if pd.notna(row["description"]):
        parts.append(row["description"])
    return " ".join(parts)

pub_long["raw_text"] = pub_long.apply(combine_text, axis=1)
#pub_long[["title", "description", "raw_text"]].head(3)

## (3) Create subject benchmarks

In [5]:
# Extract classification codes from subject field
CODE_PATTERN = re.compile(r"^(\d{3})\s")

def extract_subject_codes(subject_str):
    if pd.isna(subject_str):
        return []
    tags = [t.strip() for t in subject_str.split("|") if t.strip()]
    return [m.group(1) for t in tags if (m := CODE_PATTERN.match(t))]

pub_long["subject_codes"] = pub_long["subject"].apply(extract_subject_codes)

In [6]:
# Create subject-code shares per lab (benchmark for text analysis)

# One row per (labgroupid, code) occurrence, exploding each pub's list of codes
codes_long = pub_long[["labgroupid", "subject_codes"]].explode("subject_codes")
codes_long = codes_long.dropna(subset=["subject_codes"])

# Count how many of each lab's publications carry each code...
code_counts = pd.crosstab(codes_long["labgroupid"], codes_long["subject_codes"])
# ...then divide by that lab's total publication count to get a share
n_pubs_per_lab_tmp = pub_long.groupby("labgroupid").size()
subject_code_shares = code_counts.div(n_pubs_per_lab_tmp, axis=0).fillna(0)
subject_code_shares.columns = [f"subj_code_{c}" for c in subject_code_shares.columns]
subject_code_shares = subject_code_shares.reset_index()  # labgroupid back to a normal column, for merging

print(f"Subject-code share columns built: {subject_code_shares.shape[1] - 1}")
subject_code_shares.head(3)

Subject-code share columns built: 28


,labgroupid,subj_code_000,subj_code_070,subj_code_150,subj_code_170,subj_code_180,subj_code_300,subj_code_320,subj_code_330,subj_code_340,...,subj_code_580,subj_code_590,subj_code_600,subj_code_610,subj_code_630,subj_code_790,subj_code_890,subj_code_900,subj_code_910,subj_code_950
0,104,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0
1,131,0.0,0.0,0.0,0.142857,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.984127,0.0,0.0,0.0,0.0,0.000000,0.0
2,133,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.144444,0.055556,0.0,0.088889,0.0,0.0,0.0,0.0,0.022222,0.0


## (5) Merge subject shares with lab-level data to get BL energy use and control vars

In [7]:
# Restrict to BL observations
bl_data = labs[labs["survey"] == "BL"].copy()

# Keep only energy and control vars cols
bl_data = bl_data[["labgroupid", "annual_electricity_total", "no_researchers", 
                   "faculty", "institute_id", "treated"]].copy()

In [8]:
# Merge publications with BL data
subject_code_shares["labgroupid"] = subject_code_shares["labgroupid"].astype(int)
bl_data["labgroupid"] = bl_data["labgroupid"].astype(int)
merged_df = pd.merge(bl_data, subject_code_shares, on="labgroupid", how="inner")

## (6) Save outputs

In [9]:
pub_out_cols = ["title", "description", "date", "n_matched_labgroupids", "labgroupid", "raw_text"]
pub_long[pub_out_cols].to_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "publications_long.csv",
    index = False)

merged_df.to_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "lab_level_subject_shares.csv", 
    index=False)